### MCS Quickstart (v0.1)

This notebook demonstrates the "golden path" of the Minimal Community Standard (MCS):

1. Load the five YAML specifications (dataset/split/embedding/train/execution)
2. Validate them (schema + semantic + cross-spec consistency)
3. Generate a deterministic run identifier (`run_id`)
4. Register the run and specs into a local `.mcs_registry/`

The goal is not to train models, but to show how MCS externalizes *workflow assumptions*
in a verifiable and comparable way.

In [1]:
from pathlib import Path
import json

import mcs
from mcs.validation import summarize_by_severity, format_issues

#### 1) Locate examples

We use the repository-level `examples/` folder, which contains:
- `dataset.yaml`
- `split.yaml`
- `embedding.yaml`
- `train_classification.yaml`
- `train_regression.yaml`
- `execution.yaml`
- `execution_gpu.yaml`

In [2]:
repo_root = Path.cwd().parents[0]  # assumes running from notebooks/
examples_dir = repo_root / "examples"
examples_dir

PosixPath('/home/david/Desktop/umag_projects/MinimalCommunityStandar_v0.1/examples')

In [3]:
assert examples_dir.exists(), f"examples/ directory not found at: {examples_dir}"

### 2) Run MCS pack: classification + GPU execution

This will:
 - validate all 5 specs
 - produce a `ProvenanceRecord`
 - write registry artifacts in `.mcs_registry/`

In [4]:
out_cls = mcs.run_pack(
    examples_dir,
    train_file="train_classification.yaml",
    execution_file="execution_gpu.yaml",
    strict=False,                 # allow INFO/WARNING
    artifact_mode="reference",    # does not require real artifact files
    overwrite=True,
    notes="Quickstart: classification + GPU execution",
)

# %%
issues_cls = out_cls["issues"]
print("Issue summary:", summarize_by_severity(issues_cls))

# %%
# Show issues (optional)
print(format_issues(issues_cls[:25], header="First issues (up to 25):"))

# %%
record_cls = out_cls["record"]
paths_cls = out_cls["registry_paths"]

print("Run ID:", record_cls.run_id)
print("Run registry JSON:", paths_cls["run"])

# %%
# Inspect the stored run record (JSON)
run_record_path = Path(paths_cls["run"])
run_payload = json.loads(run_record_path.read_text(encoding="utf-8"))
run_payload.keys()

Issue summary: {'ERROR': 0, 'WARNING': 0, 'INFO': 10}
First issues (up to 25):
- [INFO] CONSIST_CUDA_EXEC_CPU_EMB: Execution uses CUDA but EmbeddingSpec.device='cpu'. This is valid if embedding extraction is CPU-bound, but verify intent. (path=embedding.device)
- [INFO] CONSIST_NO_GIT_COMMIT: ExecutionSpec.git_commit missing; reduces auditability across runs. (path=execution.git_commit)
- [INFO] DATASET_OUTPUT_NO_CHECKSUM: Dataset output has no checksum declared. (path=output.checksum) | hint: Optional but recommended for file-based reproducibility.
- [INFO] EMB_NO_DATASET_LINK: input_dataset_fingerprint not set (optional). (path=input_dataset_fingerprint) | hint: You can fill it after computing DatasetSpec.fingerprint().
- [INFO] EMB_OUTPUT_NO_CHECKSUM: Embedding output has no checksum declared. (path=output.checksum)
- [INFO] EXEC_DEPS_NO_CHECKSUM: dependencies.checksum not declared (recommended). (path=dependencies.checksum)
- [INFO] EXEC_NO_GIT_COMMIT: git_commit not declared (reco

dict_keys(['artifacts', 'created_at_utc', 'dataset_fingerprint', 'dataset_name', 'embedding_fingerprint', 'embedding_name', 'execution_fingerprint', 'execution_name', 'extra', 'mcs_version', 'notes', 'run_id', 'split_fingerprint', 'split_name', 'train_fingerprint', 'train_name'])

### 3) Run MCS pack: regression + CPU execution

In [5]:
out_reg = mcs.run_pack(
    examples_dir,
    train_file="train_regression.yaml",
    execution_file="execution.yaml",
    strict=False,
    artifact_mode="reference",
    overwrite=True,
    notes="Quickstart: regression + CPU execution",
)

issues_reg = out_reg["issues"]
print("Issue summary:", summarize_by_severity(issues_reg))

record_reg = out_reg["record"]
paths_reg = out_reg["registry_paths"]

print("Run ID:", record_reg.run_id)
print("Run registry JSON:", paths_reg["run"])

Issue summary: {'ERROR': 0, 'WARNING': 0, 'INFO': 9}
Run ID: dbddef9c15de8c55690cfce2d64d99039df5291eddf1686e9dca335033dbbaf7
Run registry JSON: .mcs_registry/runs/dbddef9c15de8c55690cfce2d64d99039df5291eddf1686e9dca335033dbbaf7.json


### 4) Why fingerprints matter

Each spec is fingerprinted based on its canonical content.

The overall `run_id` is derived from the combined fingerprints.

Small changes in assumptions → new fingerprints → new `run_id`.


In [6]:
specs_cls = out_cls["specs"]
fp_dataset = specs_cls["dataset"].fingerprint()
fp_split = specs_cls["split"].fingerprint()
fp_embedding = specs_cls["embedding"].fingerprint()
fp_train = specs_cls["train"].fingerprint()
fp_execution = specs_cls["execution"].fingerprint()

print("Dataset fingerprint:  ", fp_dataset[:16], "...")
print("Split fingerprint:    ", fp_split[:16], "...")
print("Embedding fingerprint:", fp_embedding[:16], "...")
print("Train fingerprint:    ", fp_train[:16], "...")
print("Execution fingerprint:", fp_execution[:16], "...")

Dataset fingerprint:   d8437ea87fe19414 ...
Split fingerprint:     78229a323ce73e69 ...
Embedding fingerprint: 67f616fde142f7c5 ...
Train fingerprint:     638392926fb0b4f0 ...
Execution fingerprint: c4e4999cd3d8f97d ...


### 5) Registry layout

 The registry stores:
 - run record (`runs/<run_id>.json`)
 - specs (`specs/<type>/<fingerprint>.json`)
 - (optional) artifacts under `artifacts/<run_id>/`

In [7]:
registry_root = Path(paths_cls["run"]).parents[1]  # .../.mcs_registry
registry_root

PosixPath('.mcs_registry')

In [8]:
print("Registry root:", registry_root)
for p in ["runs", "specs", "artifacts"]:
    d = registry_root / p
    print(p, "->", "OK" if d.exists() else "MISSING", "|", d)

Registry root: .mcs_registry
runs -> OK | .mcs_registry/runs
specs -> OK | .mcs_registry/specs
artifacts -> OK | .mcs_registry/artifacts


### 6) Minimal vs enriched adoption

MCS supports incremental adoption:
 - you can start with minimal YAML files
 - progressively add provenance, filters, redundancy controls, or determinism flags

See:
 - `docs/minimal_adoption.md`
 - `docs/schema_reference.md`
 - `docs/philosophy.md`